# Daily Challenge: Building an Agent with LangGraph and the Gemini API

## BaristaBot - A Conversational Cafe Ordering System

This notebook builds a stateful cafe ordering system using **LangGraph** and **Gemini API** in Google Colab.

### What You'll Learn:
- Creating stateful applications with LangGraph
- Integrating Gemini API via LangChain
- Managing state with TypedDict and node functions
- Implementing tool-augmented conversational loops
- Conditional transitions and state management

## ⚙️ Step 0: Set Up Google API Key

Before running this notebook, you need to add your Google Gemini API key:

1. Go to [Google AI Studio](https://aistudio.google.com/app/apikey)
2. Create or copy your API key
3. Click the 🔑 **Secrets** icon in the left sidebar
4. Add a new secret named `GOOGLE_API_KEY` with your key
5. Run the cell below

In [ ]:
# For Google Colab: Import and set up secrets
from google.colab import userdata
import os

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ API key configured successfully!")
except userdata.NotebookAccessError as e:
    print("❌ API key not found. Please add GOOGLE_API_KEY to Colab Secrets.")
    print("Instructions:")
    print("1. Click the 🔑 icon in the left sidebar")
    print("2. Add GOOGLE_API_KEY from https://aistudio.google.com/app/apikey")
    raise

## Step 1: Install Required Libraries

In [ ]:
# Install required packages
!pip install -qU "langgraph==1.0.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0" "langchain-core>=0.1.0"

## Step 2: Import Required Libraries

In [ ]:
# Import core libraries
from typing import Annotated, Literal
from typing_extensions import TypedDict
from collections.abc import Iterable
from random import randint
from pprint import pprint

# LangGraph and LangChain imports
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, InjectedState
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages.ai import AIMessage
from langchain_core.messages.tool import ToolMessage
from langchain_core.tools import tool

print("✅ All imports successful!")

## Step 3: Define State and System Instructions

In [ ]:
class OrderState(TypedDict):
    """State representing the customer's order conversation."""
    
    # The chat conversation with add_messages annotation for appending
    messages: Annotated[list, add_messages]
    
    # The customer's in-progress order
    order: list[str]
    
    # Flag indicating order completion
    finished: bool


# System instructions for BaristaBot
BARISTABOT_SYSINT = (
    "system",
    "You are a BaristaBot, an interactive cafe ordering system. A human will talk to you about the "
    "available products you have and you will answer any questions about menu items (and only about "
    "menu items - no off-topic discussion, but you can chat about the products and their history). "
    "The customer will place an order for 1 or more items from the menu, which you will structure "
    "and send to the ordering system after confirming the order with the human. "
    "\n\n"
    "Add items to the customer's order with add_to_order, and reset the order with clear_order. "
    "To see the contents of the order so far, call get_order (this is shown to you, not the user) "
    "Always confirm_order with the user (double-check) before calling place_order. Calling confirm_order will "
    "display the order items to the user and returns their response to seeing the list. Their response may contain modifications. "
    "Always verify and respond with drink and modifier names from the MENU before adding them to the order. "
    "If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect. "
    "You only have the modifiers listed on the menu. "
    "Once the customer has finished ordering items, Call confirm_order to ensure it is correct then make "
    "any necessary updates and then call place_order. Once place_order has returned, thank the user and "
    "say goodbye!"
)

WELCOME_MSG = "🤖 Welcome to BaristaBot cafe! How may I serve you today?"

print("✅ State and instructions defined!")

## Step 4: Initialize Gemini Model

In [ ]:
# Initialize Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-latest")
print("✅ Gemini model initialized!")

## Step 5: Define Menu Tool (Stateless)

In [ ]:
@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    return """
    ☕ MENU:
    
    Coffee Drinks:
    • Espresso
    • Americano
    • Cold Brew

    Coffee Drinks with Milk:
    • Latte
    • Cappuccino
    • Cortado
    • Macchiato
    • Mocha
    • Flat White

    Tea Drinks:
    • English Breakfast Tea
    • Green Tea
    • Earl Grey

    Tea Drinks with Milk:
    • Chai Latte
    • Matcha Latte
    • London Fog

    Other Drinks:
    • Steamer
    • Hot Chocolate

    Modifiers:
    • Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free (default: whole)
    • Espresso shots: Single, Double, Triple, Quadruple (default: Double)
    • Caffeine: Decaf, Regular (default: Regular)
    • Hot-Iced: Hot, Iced (default: Hot)
    • Sweeteners: vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
    • Special requests: extra hot, one pump, half caff, extra foam, etc.

    Notes:
    • 'Dirty' = add espresso shot to non-coffee drinks (e.g., Dirty Chai Latte)
    • Soy milk is OUT OF STOCK today
    """

print("✅ Menu tool defined!")

## Step 6: Define Ordering Tools (Stateful)

In [ ]:
# Ordering tools with empty implementations (handled by order_node)

@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Adds the specified drink to the customer's order, including any modifiers.

    Returns:
      The updated order in progress.
    """
    pass


@tool
def confirm_order() -> str:
    """Asks the customer if the order is correct.

    Returns:
      The user's free-text response.
    """
    pass


@tool
def get_order() -> str:
    """Returns the users order so far. One item per line."""
    pass


@tool
def clear_order() -> str:
    """Removes all items from the user's order."""
    pass


@tool
def place_order() -> int:
    """Sends the order to the barista for fulfillment.

    Returns:
      The estimated number of minutes until the order is ready.
    """
    pass

print("✅ Ordering tools defined!")

## Step 7: Define Node Functions

In [ ]:
def chatbot_with_welcome_msg(state: OrderState) -> OrderState:
    """The chatbot with tools and welcome message."""
    defaults = {"order": [], "finished": False}

    if state["messages"]:
        # Continue conversation with Gemini model
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        # Start with welcome message
        new_output = AIMessage(content=WELCOME_MSG)

    return defaults | state | {"messages": [new_output]}


def human_node(state: OrderState) -> OrderState:
    """Display the last model message to the user, and receive their input."""
    last_msg = state["messages"][-1]
    print(f"\n🤖 BaristaBot: {last_msg.content}")
    print()

    user_input = input("👤 You: ")

    # Check if user wants to quit
    if user_input.lower() in {"q", "quit", "exit", "goodbye"}:
        state["finished"] = True

    return state | {"messages": [("user", user_input)]}


def order_node(state: OrderState) -> OrderState:
    """Handle stateful ordering operations."""
    msgs = state.get("messages", [])
    if not msgs:
        return state
    
    tool_msg = msgs[-1]
    if not hasattr(tool_msg, "tool_calls"):
        return state
    
    order = state.get("order", [])
    outbound_msgs = []
    order_placed = False

    for tool_call in tool_msg.tool_calls:

        if tool_call["name"] == "add_to_order":
            # Add item to order with modifiers
            modifiers = tool_call["args"].get("modifiers", [])
            modifier_str = ", ".join(modifiers) if modifiers else "no modifiers"
            order.append(f'{tool_call["args"]["drink"]} ({modifier_str})')
            response = "\n".join(order)

        elif tool_call["name"] == "confirm_order":
            # Display order to user and get confirmation
            print("\n" + "="*40)
            print("📋 ORDER CONFIRMATION")
            print("="*40)
            if not order:
                print("  (no items)")
            else:
                for i, drink in enumerate(order, 1):
                    print(f"  {i}. {drink}")
            print("="*40)
            response = input("Is this correct? (yes/no or modifications): ")

        elif tool_call["name"] == "get_order":
            # Return current order (for model internal state)
            response = "\n".join(order) if order else "(no items in order)"

        elif tool_call["name"] == "clear_order":
            # Clear the entire order
            order.clear()
            response = "Order cleared."

        elif tool_call["name"] == "place_order":
            # Place the order
            order_text = "\n".join(order)
            print("\n" + "="*40)
            print("✅ SENDING ORDER TO KITCHEN!")
            print("="*40)
            print(f"Order:\n{order_text}")
            order_placed = True
            response = randint(3, 8)  # ETA in minutes
            print(f"⏱️  Estimated time: {response} minutes")
            print("="*40 + "\n")

        else:
            raise NotImplementedError(f'Unknown tool call: {tool_call["name"]}')

        # Record tool results
        outbound_msgs.append(
            ToolMessage(
                content=str(response),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": outbound_msgs, "order": order, "finished": order_placed}

print("✅ Node functions defined!")

## Step 8: Define Conditional Edge Functions

In [ ]:
def maybe_exit_human_node(state: OrderState) -> Literal["chatbot", END]:
    """Route to chatbot or exit based on user input."""
    if state.get("finished", False):
        return END
    else:
        return "chatbot"


def maybe_route_to_tools(state: OrderState) -> Literal["tools", "ordering", "human", END]:
    """Route based on tool calls and state."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found: {state}")

    msg = msgs[-1]

    # Exit if order placed
    if state.get("finished", False):
        return END

    # Route to appropriate tool handler
    if hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        # Check if it's a menu/auto tool or ordering tool
        if any(
            tool["name"] in tool_node.tools_by_name.keys()
            for tool in msg.tool_calls
        ):
            return "tools"
        else:
            return "ordering"
    else:
        return "human"

print("✅ Conditional edge functions defined!")

## Step 9: Build the Complete Graph

In [ ]:
# Set up tools
auto_tools = [get_menu]
tool_node = ToolNode(auto_tools)

order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]

# Bind all tools to LLM
llm_with_tools = llm.bind_tools(auto_tools + order_tools)

# Build graph
graph_builder = StateGraph(OrderState)

# Add nodes
graph_builder.add_node("chatbot", chatbot_with_welcome_msg)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

# Add edges
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
graph_builder.add_conditional_edges("human", maybe_exit_human_node)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("ordering", "chatbot")

# Compile graph
graph_with_order_tools = graph_builder.compile()

print("✅ Graph compiled successfully!")

## Step 10: Visualize the Graph (Optional)

In [ ]:
# Visualize the graph structure
try:
    from IPython.display import Image, display
    display(Image(graph_with_order_tools.get_graph().draw_mermaid_png()))
except ImportError:
    print("Graph visualization requires IPython. Skipping visualization.")
except Exception as e:
    print(f"Could not visualize graph: {e}")

## Step 11: Run the BaristaBot System

### Example Interactions:
1. **Order a drink**: "I'd like a latte with oat milk"
2. **Ask about menu**: "What teas do you have?"
3. **Modify order**: "Can I also add an espresso?"
4. **Check order**: Ask the bot to show your current order
5. **Special requests**: "extra hot, no foam"
6. **Quit**: Type `q` or `quit` to exit

### Expected Flow:
- Bot greets you
- You describe what you want
- Bot adds items to order
- Bot confirms order with you
- You approve or request changes
- Bot places order
- Bot says goodbye and exits

In [ ]:
# Run the complete ordering system
config = {"recursion_limit": 100}

print("\n" + "="*60)
print("☕ BARISTABOT CAFE ORDERING SYSTEM ☕")
print("="*60)
print(f"Type 'q' to quit at any time")
print("="*60)

state = graph_with_order_tools.invoke({"messages": []}, config)

print("\n" + "="*60)
print("✅ Order session complete!")
print("="*60)

## Step 12: Display Final State (Optional)

In [ ]:
# Display final state details
print("\n--- Final State Summary ---")
print(f"Order placed: {'✅ Yes' if state.get('finished', False) else '❌ No'}")
print(f"Final order items: {len(state.get('order', []))}")
if state.get('order'):
    for i, item in enumerate(state.get('order', []), 1):
        print(f"  {i}. {item}")
print(f"Total messages exchanged: {len(state.get('messages', []))}")

## 🔧 Troubleshooting

### Issue: "API key not found"
- ✅ Go to 🔑 Secrets in left sidebar
- ✅ Add `GOOGLE_API_KEY` from https://aistudio.google.com/app/apikey
- ✅ Re-run the Setup cell

### Issue: Import errors
- ✅ Ensure all packages installed in Step 1
- ✅ Restart runtime: Runtime → Restart session

### Issue: Input() not working
- ✅ Google Colab input() should work - make sure to run cell sequentially
- ✅ If not, use text input widgets (optional)

### Issue: Rate limiting
- ✅ Add delays between requests if hitting Gemini API limits
- ✅ Check your API quota at https://aistudio.google.com/app/usage